# LangChain + Clembench experiment constructor

**Single configurable notebook** for running LangChain agents on clembench games.

**Only Cell 2 (Config) needs to be changed between experiments.**

| Variable | Description |
|---|---|
| `GAME` | any clembench game (e.g., `taboo`, `referencegame`, etc.|
| `MODEL` | any model (e.g.,`qwen`, `gpt-4o-mini`) |
| `AGENT_TYPE` | a LangChain agent (choose from below or create a custom one) |
| `RUN_ID` | any string — used as the results folder name |
| `NUM_EPISODES` | integer |
| `SINGLE_PASS` | `True` = one pass through instances, `False` = cycle infinitely |

In [46]:
# ── CONFIG ──────────────────────────────────────────────────────────
GAME         = "referencegame"   
MODEL        = "qwen3-vl:30b-a3b-instruct"            
AGENT_TYPE   = "CoreToolsAgent"  
RUN_ID       = "my-run-1"
NUM_EPISODES = 15
SINGLE_PASS  = True
# ───────────────────────────────────────────────────────────────────────

## 1. Preparation

In [47]:
import os

CLEMBENCH_HOME = r"C:\Users\white\Desktop\agents_experiments\clembench_v3"
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [48]:
# uncomment and run once to install dependencies, then comment out again
# %pip install -r $CLEMBENCH_HOME/requirements.txt
# %pip install --upgrade ipywidgets jupyter_client clemcore

In [49]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s {GAME}

clem 3.5.0
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'referencegame'}'
referencegame:
 	Reference Game between two agents where one has to describe one of
	three grids and the other has to guess which one it is.


In [50]:
import json

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from playpen.agents import ClemAgent, ClemObservation
from clemcore.clemgame import env, episode_results_folder_callbacks

from clemcore.backends import ModelRegistry
from clemcore.backends import KeyRegistry

In [51]:
#register the model if necessary

#registry = ModelRegistry.register("insert model name", backend="insert backend",
 #                                 model_id="insert model id")
#registry.get_first_model_spec_that_unify_with("insert model id")

In [52]:
#register the API key and url if necessary

#API_KEY = "insert key"
#ORGANIZATION = "insert organization"
#BASE_URL ="insert url" 

#KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

In [53]:
def create_model(
    model_name: str,
    registry_path: str = "model_registry.json",
    key_path: str = "key.json",
    temperature: float = 0,
    max_tokens: int = 300,) -> ChatOpenAI:
    """Returns a configured ChatOpenAI instance by looking up model_name in model_registry.json.
    Raises ValueError if the model or its required backend credentials are not found.
    """
    with open(registry_path) as f:
        registry = json.load(f)

    entry = next((e for e in registry if e["model_name"] == model_name), None)
    if entry is None:
        raise ValueError(f"Model {model_name!r} not found in {registry_path}.")

    backend = entry["backend"]

    with open(key_path) as f:
        keys = json.load(f)

    if backend not in keys:
        raise ValueError(
            f"Backend {backend!r} (required by model {model_name!r}) "
            f"not found in {key_path}."
        )

    credentials = keys[backend]

    return ChatOpenAI(
        model=entry["model_id"],
        base_url=credentials["base_url"],
        api_key=credentials["api_key"],
        temperature=temperature,
        max_tokens=max_tokens,
    )

## 2. Agent constructor

In [54]:
# Subagent prompt used by MyAgenticPlayer 
_SUBAGENT_PROMPT = """
You are given a small piece of text which contains gameplay rules. You need to extract the necessary tags
(often written in CAPITAL LETTERS), so that the player can use them for the answer.
Do not output any text apart from the tag(s). Example IO pair:

INPUT:
Let's play a guessing game! Your task is to answer the other player's questions. Based on your knowledge
of the word: $TARGET WORD$, respond to the following questions or guesses. Limit your response to only
'yes' or 'no' with no explanation or other words. Never reveal the answer in your response.

You must reply using the format below and DO NOT ADD ANY TEXT OTHER THAN THIS:

ANSWER: <some text>

Target Word: $TARGET WORD$

OUTPUT:
ANSWER:

If you identified no tags, please return NO TAG as an answer.
"""


# Agent definitions 

class CoreToolsAgent(ClemAgent):
    """Agent with remember / recall / observe / get_observations tools."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self.observations = []

        tools = [
            self._remember(),
            self._recall(),
            self._observe(),
            self._get_observations(),
        ]

        system_prompt = """You're a professional game player with memory tools.

STRATEGY:
1. On FIRST turn: understand the rules, store key info with remember()
2. After 1-2 tool calls, you MUST give your final answer.
Do NOT keep calling tools."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self.observations.clear()

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _remember(self):
        store = self.store
        @tool
        def remember(key: str, value: str) -> str:
            """
            Store any important information.

            Examples:
                remember("goal", "describe the target without forbidden words")
                remember("format", "CLUE: <text>")
                remember("target", "first grid")
                remember("forbidden", "cat, dog, pet")
            """
            store[key] = value
            return f"Stored: {key} = {value}"
        return remember

    def _recall(self):
        store = self.store
        @tool
        def recall(key: str = "") -> str:
            """
            Retrieve stored information.

            Args:
                key: Specific key, or empty for everything
            """
            if not store:
                return "Memory empty."
            if key and key in store:
                return f"{key}: {store[key]}"
            return "\n".join(f"- {k}: {v}" for k, v in store.items())
        return recall

    def _observe(self):
        observations = self.observations
        @tool
        def observe(observation: str) -> str:
            """
            Note something important you noticed.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue mentions 'round shape'")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe

    def _get_observations(self):
        observations = self.observations
        @tool
        def get_observations() -> str:
            """Get all observations you've noted."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_observations

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"


class MyAgenticPlayer(ClemAgent):
    """Agent that uses an extract_tags subagent to parse game rules on first turn."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.model = model
        self.memory = InMemorySaver()
        self.base_thread_id = thread_id
        self.episode = 0

        @tool
        def extract_tags(initial_prompt: str) -> str:
            """Extract the response tags that are necessary for the player from the game rules."""
            subagent = create_agent(model=self.model, tools=[], system_prompt=_SUBAGENT_PROMPT)
            result = subagent.invoke({"messages": [{"role": "user", "content": initial_prompt}]})
            final = next(m for m in reversed(result["messages"]) if isinstance(m, AIMessage))
            return final.content

        self.agent = create_agent(
            model=self.model,
            tools=[extract_tags],
            checkpointer=self.memory,
            system_prompt=(
                "You're going to play a game. You're a professional agent game player. "
                "You have a helpful tool extract_tags that identifies the required response format. "
                "On your first turn, call extract_tags and use the result to format all future responses."
            ),
        )

    def reset(self):
        super().reset()
        self.episode += 1

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}},
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

## 3. Experiment setup

In [55]:
def make_agent(agent_type: str, model: ChatOpenAI, thread_id: str) -> ClemAgent:
    """Instantiate an agent by name."""
    if agent_type == "CoreToolsAgent":
        return CoreToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "MyAgenticPlayer":
        return MyAgenticPlayer(model=model, thread_id=thread_id)
    else:
        raise ValueError(f"Unknown agent type: {agent_type!r}.")

In [56]:
callbacks = episode_results_folder_callbacks(
    run_dir=RUN_ID,
    result_dir_path="playpen-records",
    player_model_infos=f"{AGENT_TYPE}-{MODEL}",
)

game_env = env(GAME, single_pass=SINGLE_PASS, callbacks=callbacks)
#removed reset

print("roles:", game_env.unwrapped.game_benchmark.game_spec["roles"])

2026-02-28 21:37:11,126 - clemcore.cli - INFO - Found '1' game matching the game_selector="referencegame"
2026-02-28 21:37:11,129 - clemcore.cli - INFO - {
  "game_name": "referencegame",
  "description": "Reference Game between two agents where one has to describe one of three grids and the other has to guess which one it is.",
  "main_game": "referencegame",
  "players": 2,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "0.9",
    "1.0",
    "1.5",
    "2.0",
    "3.0"
  ],
  "regression": "large",
  "roles": [
    "Instruction Giver",
    "Instruction Follower"
  ],
  "game_path": "C:\\Users\\white\\Desktop\\agents_experiments\\clembench_v3\\referencegame"
}
2026-02-28 21:37:11,132 - clemcore.run - INFO - Loading game benchmark for referencegame
2026-02-28 21:37:11,141 - clemcore.run - INFO - Loading game benchmark for referencegame took: 0:00:00.006172
2026-02-28 21:37:11,166 - clemcore.run - INFO - Prepared instance queue for referencegame using 5 experimen

In [57]:
model = create_model(MODEL)

if GAME == "textmapworld":
    guesser = make_agent(AGENT_TYPE, model, thread_id="guesser")
    learner_agents = [guesser]
    agent_mapping = {"player_0": guesser, "player_1": None}  # player_1 refreshed in loop
else:
    describer = make_agent(AGENT_TYPE, model, thread_id="describer")
    guesser   = make_agent(AGENT_TYPE, model, thread_id="guesser")
    learner_agents = [describer, guesser]
    agent_mapping  = {"player_0": describer, "player_1": guesser}

print("Agent mapping:", {k: type(v).__name__ for k, v in agent_mapping.items()})

Agent mapping: {'player_0': 'CoreToolsAgent', 'player_1': 'CoreToolsAgent'}


## 4. Run game

In [58]:
import json as _json
from pathlib import Path
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


#TODO: make memory printing / saving OPTIONAL because not all planned setups need memory

def serialize_messages(msgs: list) -> list:
    """Convert LangChain messages to plain dicts for JSON serialization."""
    out = []
    for m in msgs:
        out.append({
            "type": type(m).__name__,
            "content": m.content,
            "tool_calls": getattr(m, "tool_calls", []),
        })
    return out

memory_log_dir = Path("playpen-records") / RUN_ID / GAME / "memory_logs"   #NB: kept separately from the results!
memory_log_dir.mkdir(parents=True, exist_ok=True)                   #create the directory if absent

all_episodes_data = []

for episode in range(NUM_EPISODES):
    game_env.reset()
    for agent in learner_agents:
        agent.reset()

    # textmapworld: built-in describer is recreated on every reset() — refresh the reference
    #TODO: smth has to be done about singleplayer games 
    if GAME == "textmapworld":
        agent_mapping["player_1"] = game_env.unwrapped.game_master.describer

    episode_memory_log = []  # collects memory snapshots for this episode

    context_response_pairs = []
    for step_idx, agent_id in enumerate(game_env.agent_iter()):
        context, reward, termination, truncation, info = game_env.last()
        response = None if (termination or truncation) else agent_mapping[agent_id](context)
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

        # snapshot memory for the agent that just acted
        agent = agent_mapping.get(agent_id)
        if agent is not None and hasattr(agent, "get_memory_snapshot"):
            msgs = agent.get_memory_snapshot()
            snapshot = {
                "step": step_idx,
                "agent_id": agent_id,
                "thread_id": agent.base_thread_id,
                "messages": serialize_messages(msgs),
            }
            episode_memory_log.append(snapshot)
            print(f"  [memory:{agent.base_thread_id}] {len(msgs)} messages: "
                  + " | ".join(f"{type(m).__name__}({m.content[:40]!r})" for m in msgs))

    # write memory log for this episode 
    log_path = memory_log_dir / f"episode_{episode + 1:04d}.json"
    with open(log_path, "w") as f:
        _json.dump(episode_memory_log, f, indent=2, default=str)

    all_episodes_data.append(context_response_pairs)
    print(f"Episode {episode + 1}/{NUM_EPISODES} completed — {len(context_response_pairs)} steps")

print(f"\nAll episodes done. Memory logs written to: {memory_log_dir}")

  [memory:describer] 11 messages: HumanMessage('You are given three grids, where each of') | AIMessage('') | ToolMessage('Stored: target = first grid') | ToolMessage('Stored: format = Expression: <text>') | ToolMessage('Noted: Target grid has a vertical line o') | ToolMessage('Noted: Distractor 1 has Xs in the third ') | ToolMessage('Noted: Distractor 2 has Xs in alternatin') | AIMessage('') | ToolMessage('target: first grid') | ToolMessage('format: Expression: <text>') | AIMessage('Expression: a grid with a vertical line ')
  [memory:guesser] 2 messages: HumanMessage('You are given three grids, where each of') | AIMessage('Answer: first')
  [memory:describer] 11 messages: HumanMessage('You are given three grids, where each of') | AIMessage('') | ToolMessage('Stored: target = first grid') | ToolMessage('Stored: format = Expression: <text>') | ToolMessage('Noted: Target grid has a vertical line o') | ToolMessage('Noted: Distractor 1 has Xs in the third ') | ToolMessage('Noted: Distracto

In [59]:
# Display the last episode's steps
last_episode = all_episodes_data[-1]
print(f"Last episode: {len(last_episode)} steps")
print("-" * 60)
for idx, (agent_id, context, response, reward) in enumerate(last_episode):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"  Agent({agent_id}) <- Context: {context}")
    print(f"  Agent({agent_id}) -> Response: {response}")
    print("-" * 60)

Last episode: 4 steps
------------------------------------------------------------
Step 0 / Reward 0.00:
  Agent(player_0) <- Context: {'role': 'user', 'content': 'You are given three grids, where each of them is 5 by 5 in size.\nGrids have empty cells marked with "▢" and filled cells marked with "X".\nYour task is to generate a referring expression that best describes the target grid while distinguishing it from the two other distractor grids.\nThe first grid is the target grid, and the following two grids are the distractors.\n\nTarget grid:\n\nX X □ X X\nX X □ X X\nX X □ X X\nX X □ X X\nX X □ X X\n\nDistractor grid 1:\n\nX □ □ X X\nX □ □ X X\nX □ □ X X\nX □ □ X X\nX □ □ X X\n\nDistractor grid 2:\n\n□ □ □ X X\n□ □ □ X X\n□ □ □ X X\n□ □ □ X X\n□ □ □ X X\n\nInstruction: Describe the target grid.\nGenerate the referring expression starting with the tag "Expression: " for the given target grid. Omit any other text.'}
  Agent(player_0) -> Response: Expression: The grid with a vertical lin

## 5. After the game

In [60]:
results_dir = callbacks.callbacks[0].results_folder.results_dir_path
run_dir     = callbacks.callbacks[0].results_folder.run_dir
print(f"Results saved to: {results_dir}")
print(f"Run dir:          {run_dir}")
print()
print("To score results:")
print(f"  clem score -g {GAME} -r playpen-records") #or -r PATH_TO_FOLDER
print(f"  clem eval -r playpen-records")   #or -r PATH_TO_FOLDER

Results saved to: playpen-records
Run dir:          my-run-1

To score results:
  clem score -g referencegame -r playpen-records
  clem eval -r playpen-records
